# iSign Track 1 · Notebook 2 — train + evaluate (exact paper recipe)

Reproduces **arXiv 2609.12993** with the authors' own architecture and settings:

| | Paper setting |
|---|---|
| Input | 33 body + 2×21 hand landmarks × (x, y, z, conf) = **300-d** per frame; **+ motion** `x_t − x_{t−1}` → **600-d** |
| Normalisation | per-example z-score, pad/truncate to **500 frames** |
| Pose encoder | Linear(pose_dim→256) → ReLU → Dropout(0.1) → Linear(256→d_model) |
| Language model | pretrained **T5** (`inputs_embeds` = pose embeddings), fully fine-tuned |
| Optimiser | AdamW, lr **5e-4 constant**, batch **16**, grad-accum **2**, clip **1.0**, **fp16** |
| Epochs | **10** (t5-small, t5-base, t5-small-motion), **3** (t5-large) |
| Model selection | lowest validation loss, checked every 500 steps |
| Decoding | beam **4**, length penalty **2.0**, max **128** tokens |
| Test split | 10% held out, `train_test_split(random_state=42)` → **1,887** examples |

Paper results on that split (BLEU on a 0–100 scale): t5-small **0.189**, t5-base 0.154, t5-large 0.132,
**t5-small + Motion 0.298** (best).

### Settings
* **Add Input** → your dataset **`isign-partad-features`** (from notebook 1).
* **Accelerator: GPU T4 x2** (use T4, not P100: current PyTorch builds may not support the
  P100). Only one GPU is used, like the paper's single-GPU code.
* **Internet: On** (downloads pretrained T5 from HuggingFace).
* **Save Version → Save & Run All (Commit)** so it keeps running after you close the tab.
  t5-small-motion ≈ 45–90 min.

In [ ]:
!pip install -q "sacrebleu==2.6.0" "rouge-score==0.1.2" sentencepiece

### Source files (identical to the locally tested versions)

In [ ]:
%%writefile paper_pose_utils.py
# VERBATIM copy of inference/pose_utils.py from huggingface.co/manavdhamecha77/iSign-t5-pose-to-text
# (paper arXiv 2609.12993). Do not edit: this is the exact preprocessing the paper used.
"""Utilities for reading and preprocessing iSign binary .pose files."""

import logging
import json
from typing import List, Optional, Sequence, Tuple

import numpy as np

logger = logging.getLogger(__name__)

DEFAULT_COMPONENTS = (
    "POSE_LANDMARKS",
    "LEFT_HAND_LANDMARKS",
    "RIGHT_HAND_LANDMARKS",
)


def load_pose_file(
    pose_path: str,
    components: Optional[Sequence[str]] = DEFAULT_COMPONENTS,
    include_confidence: bool = True,
) -> np.ndarray:

    try:
        with open(pose_path, "rb") as f:
            raw = f.read()

        if raw.lstrip().startswith((b"{", b"[")):
            payload = json.loads(raw.decode("utf-8"))
            frames = (
                payload.get("data", [])
                if isinstance(payload, dict)
                else payload
            )

            rows = [
                np.asarray(
                    frame.get("keypoints", []),
                    dtype=np.float32,
                ).reshape(-1)
                for frame in frames
            ]

            return (
                np.asarray(rows, dtype=np.float32)
                if rows
                else np.array([])
            )

    except (
        json.JSONDecodeError,
        UnicodeDecodeError,
        FileNotFoundError,
        OSError,
    ) as exc:
        logger.error(
            "Failed to read pose file %s: %s",
            pose_path,
            exc,
        )
        return np.array([])

    try:
        from pose_format import Pose

        pose = Pose.read(raw)

    except (
        FileNotFoundError,
        ImportError,
        ValueError,
        OSError,
    ) as exc:
        logger.error(
            "Failed to load pose file %s: %s",
            pose_path,
            exc,
        )
        return np.array([])

    data = np.asarray(
        pose.body.data,
        dtype=np.float32,
    )

    confidence = np.asarray(
        pose.body.confidence,
        dtype=np.float32,
    )

    if data.ndim != 4 or data.shape[1] == 0:
        logger.warning(
            "Unexpected pose shape in %s: %s",
            pose_path,
            data.shape,
        )
        return np.array([])

    data = data[:, 0]

    confidence = (
        confidence[:, 0]
        if confidence.ndim == 3
        else None
    )

    wanted = (
        set(components)
        if components is not None
        else None
    )

    selected = []

    offset = 0

    for component in pose.header.components:

        count = len(component.points)

        if (
            wanted is None
            or component.name in wanted
        ):

            component_data = data[
                :,
                offset:offset + count,
                :,
            ]

            if (
                include_confidence
                and confidence is not None
            ):

                component_data = np.concatenate(
                    [
                        component_data,
                        confidence[
                            :,
                            offset:offset + count,
                            None,
                        ],
                    ],
                    axis=2,
                )

            selected.append(component_data)

        offset += count

    if not selected:

        logger.warning(
            "No requested components found in %s",
            pose_path,
        )

        return np.array([])

    return np.concatenate(
        selected,
        axis=1,
    ).reshape(
        data.shape[0],
        -1,
    )


def normalize_pose(
    pose: np.ndarray,
    method: str = "zscore",
) -> np.ndarray:

    if pose.size == 0:
        return pose

    if method == "zscore":

        mean = np.nanmean(
            pose,
            axis=0,
            keepdims=True,
        )

        std = np.nanstd(
            pose,
            axis=0,
            keepdims=True,
        )

        return np.nan_to_num(
            (pose - mean)
            / np.where(
                std == 0,
                1.0,
                std,
            )
        )

    if method == "minmax":

        min_val = np.nanmin(
            pose,
            axis=0,
            keepdims=True,
        )

        max_val = np.nanmax(
            pose,
            axis=0,
            keepdims=True,
        )

        return np.nan_to_num(
            (pose - min_val)
            / np.where(
                max_val == min_val,
                1.0,
                max_val - min_val,
            )
        )

    raise ValueError(
        f"Unknown normalization method: {method}"
    )


def pad_or_truncate_pose(
    pose: np.ndarray,
    max_length: int,
    pad_value: float = 0.0,
) -> Tuple[np.ndarray, int]:

    if pose.size == 0:

        width = (
            pose.shape[1]
            if pose.ndim > 1
            else 0
        )

        return (
            np.full(
                (max_length, width),
                pad_value,
                dtype=np.float32,
            ),
            0,
        )

    length = min(
        pose.shape[0],
        max_length,
    )

    padded = np.full(
        (max_length, pose.shape[1]),
        pad_value,
        dtype=pose.dtype,
    )

    padded[:length] = pose[:length]

    return padded, length


def add_motion_features(
    pose: np.ndarray,
) -> np.ndarray:
    """
    Concatenate frame-wise velocity with pose features.

    velocity[t] = pose[t] - pose[t - 1]

    First frame velocity is zero.
    """

    if pose.size == 0:
        return pose

    velocity = np.zeros_like(
        pose,
        dtype=np.float32,
    )

    velocity[1:] = (
        pose[1:] - pose[:-1]
    )

    return np.concatenate(
        [
            pose,
            velocity,
        ],
        axis=1,
    )


def preprocess_pose(
    pose_path: str,
    max_length: int,
    normalization: str = "zscore",
    components: Optional[
        Sequence[str]
    ] = DEFAULT_COMPONENTS,
    include_confidence: bool = True,
    pose_dim: int = 600,
    use_motion: bool = True,
) -> Tuple[np.ndarray, int]:

    pose = load_pose_file(
        pose_path,
        components,
        include_confidence,
    )

    base_pose_dim = (
        pose_dim // 2
        if use_motion
        else pose_dim
    )

    if pose.size == 0:

        return (
            np.zeros(
                (
                    max_length,
                    pose_dim,
                ),
                dtype=np.float32,
            ),
            0,
        )

    if pose.shape[1] < base_pose_dim:

        pose = np.pad(
            pose,
            (
                (0, 0),
                (
                    0,
                    base_pose_dim
                    - pose.shape[1],
                ),
            ),
        )

    elif pose.shape[1] > base_pose_dim:

        pose = pose[
            :,
            :base_pose_dim,
        ]

    pose = normalize_pose(
        pose,
        normalization,
    )

    if use_motion:

        pose = add_motion_features(
            pose,
        )

    return pad_or_truncate_pose(
        pose,
        max_length,
    )


def create_attention_mask(
    actual_lengths: List[int],
    max_length: int,
) -> np.ndarray:

    mask = np.zeros(
        (
            len(actual_lengths),
            max_length,
        ),
        dtype=np.int64,
    )

    for index, length in enumerate(
        actual_lengths
    ):

        mask[
            index,
            :length,
        ] = 1

    return mask

In [ ]:
%%writefile paper_model.py
# VERBATIM copy of inference/model.py from huggingface.co/manavdhamecha77/iSign-t5-pose-to-text
# (paper arXiv 2609.12993). Do not edit: this is the exact PoseEncoder + PoseT5 architecture.
"""
PoseT5: T5 model with pose keypoint encoder.
Maps pose keypoints -> T5 embedding space -> T5 encoder-decoder.
"""

import torch
import torch.nn as nn
from transformers import T5ForConditionalGeneration, AutoTokenizer
import logging

logger = logging.getLogger(__name__)


class PoseEncoder(nn.Module):
    """
    Encodes pose keypoints into T5 embedding space.
    
    Input: (batch_size, seq_length, pose_dim)
    Output: (batch_size, seq_length, embedding_dim)
    """
    
    def __init__(self, pose_dim: int, embedding_dim: int, hidden_dim: int = 256):
        super().__init__()
        self.pose_dim = pose_dim
        self.embedding_dim = embedding_dim
        
        # Simple MLP: pose_dim -> hidden_dim -> embedding_dim
        self.encoder = nn.Sequential(
            nn.Linear(pose_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(hidden_dim, embedding_dim),
        )
    
    def forward(self, pose: torch.Tensor) -> torch.Tensor:
        """
        Args:
            pose: (batch_size, seq_length, pose_dim)
        
        Returns:
            embeddings: (batch_size, seq_length, embedding_dim)
        """
        return self.encoder(pose)


class PoseT5(nn.Module):
    """
    T5 model with pose keypoint input.
    
    Architecture:
    - PoseEncoder: pose keypoints -> T5 embeddings
    - T5ForConditionalGeneration: standard encoder-decoder
    """
    
    def __init__(
        self,
        model_name: str,
        pose_dim: int,
        local_model_path: str = None,
    ):
        super().__init__()
        self.model_name = model_name
        self.pose_dim = pose_dim
        
        # Load T5 model
        if local_model_path:
            logger.info(f"Loading T5 from local path: {local_model_path}")
            self.t5 = T5ForConditionalGeneration.from_pretrained(local_model_path)
        else:
            logger.info(f"Loading T5 from HuggingFace: {model_name}")
            self.t5 = T5ForConditionalGeneration.from_pretrained(model_name)
        
        # Get embedding dimension from T5
        embedding_dim = self.t5.config.d_model
        
        # Create pose encoder
        self.pose_encoder = PoseEncoder(pose_dim, embedding_dim)
        
        logger.info(f"PoseT5 initialized: pose_dim={pose_dim}, embedding_dim={embedding_dim}")
    
    def forward(
        self,
        pose: torch.Tensor,
        pose_length: torch.Tensor,
        input_ids: torch.Tensor,
        attention_mask: torch.Tensor,
        labels: torch.Tensor = None,
        decoder_input_ids: torch.Tensor = None,
    ):
        """
        Forward pass for training.
        
        Args:
            pose: (batch_size, seq_length, pose_dim) - pose keypoints
            pose_length: (batch_size,) - actual length of each pose sequence
            input_ids: (batch_size, tgt_seq_length) - target text token ids
            attention_mask: (batch_size, tgt_seq_length) - target attention mask
            labels: (batch_size, tgt_seq_length) - target labels for loss computation
            decoder_input_ids: (batch_size, tgt_seq_length) - optional decoder input
        
        Returns:
            outputs: T5 model output (loss, logits, etc.)
        """
        # Encode pose to embeddings
        pose_embeddings = self.pose_encoder(pose)  # (batch_size, seq_length, d_model)
        
        # Create pose attention mask (1 for real tokens, 0 for padding)
        batch_size, seq_length, _ = pose_embeddings.shape
        device = pose_embeddings.device
        
        pose_attention_mask = (torch.arange(seq_length, device=device)[None, :] < pose_length[:, None]).long()
        
        # Forward through T5
        # Use pose embeddings as encoder input instead of text embeddings
        outputs = self.t5(
            inputs_embeds=pose_embeddings,  # Use pose embeddings directly
            attention_mask=pose_attention_mask,
            decoder_input_ids=decoder_input_ids,
            labels=labels,
        )
        
        return outputs
    
    def generate(
        self,
        pose: torch.Tensor,
        pose_length: torch.Tensor,
        max_length: int = 128,
        num_beams: int = 4,
        early_stopping: bool = True,
    ):
        """
        Generate text from pose input.
        
        Args:
            pose: (batch_size, seq_length, pose_dim)
            pose_length: (batch_size,)
            max_length: Maximum generation length
            num_beams: Beam search size
            early_stopping: Stop early if done
        
        Returns:
            generated_ids: (batch_size, max_length)
        """
        # Encode pose
        pose_embeddings = self.pose_encoder(pose)
        
        # Create attention mask
        batch_size, seq_length, _ = pose_embeddings.shape
        device = pose_embeddings.device
        
        pose_attention_mask = (torch.arange(seq_length, device=device)[None, :] < pose_length[:, None]).long()
        
        # Generate
        generated_ids = self.t5.generate(
            inputs_embeds=pose_embeddings,
            attention_mask=pose_attention_mask,
            max_length=max_length,
            num_beams=num_beams,
            early_stopping=early_stopping,
            length_penalty=2.0,
        )
        
        return generated_ids
    
    def save_pretrained(self, save_path: str):
        """Save model and encoder."""
        import os
        os.makedirs(save_path, exist_ok=True)
        
        self.t5.save_pretrained(os.path.join(save_path, "t5"))
        torch.save(self.pose_encoder.state_dict(), os.path.join(save_path, "pose_encoder.pt"))
        logger.info(f"Model saved to {save_path}")
    
    def load_pretrained(
        self,
        load_path: str,
        load_pose_encoder: bool = True,
    ):

        import os

        self.t5 = T5ForConditionalGeneration.from_pretrained(
            os.path.join(
                load_path,
                "t5",
            )
        )

        if load_pose_encoder:

            pose_encoder_path = os.path.join(
                load_path,
                "pose_encoder.pt",
            )

            self.pose_encoder.load_state_dict(
                torch.load(
                    pose_encoder_path,
                    map_location="cpu",
                )
            )

        logger.info(
            f"Model loaded from {load_path}"
        )


def create_model(
    model_name: str,
    pose_dim: int,
    local_model_path: str = None,
):
    """Factory function to create PoseT5 model."""
    return PoseT5(
        model_name=model_name,
        pose_dim=pose_dim,
        local_model_path=local_model_path,
    )

In [ ]:
%%writefile features.py
"""Turn iSign .pose bytes into the paper's raw per-frame feature matrix.

We call the paper's own `load_pose_file` so the numbers are identical to what
their training code saw: POSE(33) + LEFT_HAND(21) + RIGHT_HAND(21) landmarks,
each (x, y, z, confidence) -> (frames, 300) float32.

Everything after loading (z-score, motion, padding to 500 frames) is applied at
training time by `preprocess_array`, so one cached feature file serves both the
spatial (300-d) and motion (600-d) models.
"""

import os
import tempfile

import numpy as np

from paper_pose_utils import (
    DEFAULT_COMPONENTS,
    add_motion_features,
    load_pose_file,
    normalize_pose,
    pad_or_truncate_pose,
)


def pose_bytes_to_features(pose_bytes):
    """Exactly `load_pose_file(path, DEFAULT_COMPONENTS, include_confidence=True)`."""
    fd, path = tempfile.mkstemp(suffix=".pose")
    try:
        with os.fdopen(fd, "wb") as f:
            f.write(pose_bytes)
        return load_pose_file(path, DEFAULT_COMPONENTS, True)
    finally:
        os.remove(path)


def preprocess_array(pose, max_length=500, normalization="zscore", pose_dim=300,
                     use_motion=False):
    """Same steps, in the same order, as the paper's `preprocess_pose` after the
    file has been loaded (see paper_pose_utils.preprocess_pose)."""
    base_pose_dim = pose_dim // 2 if use_motion else pose_dim
    if pose.size == 0:
        return np.zeros((max_length, pose_dim), dtype=np.float32), 0
    if pose.shape[1] < base_pose_dim:
        pose = np.pad(pose, ((0, 0), (0, base_pose_dim - pose.shape[1])))
    elif pose.shape[1] > base_pose_dim:
        pose = pose[:, :base_pose_dim]
    pose = normalize_pose(pose, normalization)
    if use_motion:
        pose = add_motion_features(pose)
    return pad_or_truncate_pose(pose, max_length)

In [ ]:
%%writefile pipeline.py
"""Training + evaluation, reproducing arXiv 2609.12993 (Sign-Lang-Trans repo).

Faithful to the paper / their src/train.py, src/dataset.py, src/split_data.py,
src/evaluate_all.py:
  * test split  : sklearn train_test_split(test_size=0.1, random_state=42)
                  over the samples that have poses (their split_data.py)
  * val split   : torch random_split 80/10/10 of the remaining rows,
                  Generator seed 42 (their create_data_loaders); the extra 10%
                  "internal test" is unused, exactly as in their code
  * optimizer   : AdamW lr 5e-4, CONSTANT (the paper states warmup was not applied)
  * batch 16, gradient accumulation 2, grad-clip 1.0, fp16 autocast + GradScaler
  * epochs      : 10 (t5-small, t5-base, t5-small-motion), 3 (t5-large)
  * eval on val every 500 micro-batches; best_model = lowest val loss
  * decoding    : beam 4, length_penalty 2.0, max 128 tokens
  * metrics     : sacrebleu BLEU + chrF (corpus), ROUGE-1/2/L (stemmed, averaged)

Deliberate deviations (none change the model or the optimisation):
  * features are read from cached .npy files instead of re-parsing .pose files
    each epoch (numerically identical, see features.py);
  * only best_model and the final model are saved, not every 500 steps
    (their per-step checkpoints would overflow Kaggle's 20 GB output).
"""

import csv
import json
import logging
import math
import os
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch.optim import AdamW
from torch.utils.data import DataLoader, Dataset, random_split
from transformers import AutoTokenizer

from features import preprocess_array
from paper_model import PoseT5

logger = logging.getLogger("isign")

# Per-variant settings from the paper (section 4) and their configs/config_*.yaml.
VARIANTS = {
    "t5-small":        dict(base_model="t5-small", pose_dim=300, use_motion=False, epochs=10),
    "t5-small-motion": dict(base_model="t5-small", pose_dim=600, use_motion=True,  epochs=10),
    "t5-base":         dict(base_model="t5-base",  pose_dim=300, use_motion=False, epochs=10),
    "t5-large":        dict(base_model="t5-large", pose_dim=300, use_motion=False, epochs=3),
}

TRAIN_DEFAULTS = dict(
    batch_size=16, learning_rate=5e-4, gradient_accumulation_steps=2, max_grad_norm=1.0,
    mixed_precision="fp16", logging_steps=100, eval_steps=500, num_workers=4, seed=42,
    max_pose_length=500, max_text_length=128, normalization="zscore",
    beam_size=4, max_gen_length=128,
)


# ----------------------------------------------------------------------------- data

def load_annotations(csv_path, features_dir):
    """iSign CSV rows (in file order) that have a cached feature file.
    Mirrors their loader: keeps (uid, text), drops rows without a pose."""
    df = pd.read_csv(csv_path)
    if not {"uid", "text"}.issubset(df.columns):
        raise ValueError(f"expected uid,text columns in {csv_path}, got {list(df.columns)}")
    df = df[["uid", "text"]].copy()
    df["uid"] = df["uid"].astype(str)
    df["text"] = df["text"].fillna("").astype(str)
    features_dir = Path(features_dir)
    have = df["uid"].map(lambda u: (features_dir / f"{u}.npy").exists())
    return df[have].reset_index(drop=True)


def paper_test_split(df, seed=42, test_fraction=0.1):
    """Their split_data.py: sklearn train_test_split(test_size=0.1, random_state=42)."""
    from sklearn.model_selection import train_test_split
    train_df, test_df = train_test_split(df, test_size=test_fraction, random_state=seed,
                                         shuffle=True)
    return train_df.reset_index(drop=True), test_df.reset_index(drop=True)


def video_test_split(df, seed=42, test_fraction=0.1):
    """Stricter alternative: no video contributes segments to both train and test
    (what the iSign authors recommend). uid = '<video_id>-<segment>'."""
    vids = df["uid"].str.rsplit("-", n=1).str[0]
    unique = vids.drop_duplicates().sample(frac=1.0, random_state=seed).tolist()
    test_vids, n = set(), 0
    for v in unique:
        if n >= test_fraction * len(df):
            break
        test_vids.add(v)
        n += int((vids == v).sum())
    is_test = vids.isin(test_vids)
    return df[~is_test].reset_index(drop=True), df[is_test].reset_index(drop=True)


class PoseSLTDataset(Dataset):
    """Same outputs as their PoseSLTDataset, reading cached features."""

    def __init__(self, df, features_dir, tokenizer, pose_dim, use_motion, cfg):
        self.df, self.dir, self.tok = df.reset_index(drop=True), Path(features_dir), tokenizer
        self.pose_dim, self.use_motion, self.cfg = pose_dim, use_motion, cfg

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        raw = np.load(self.dir / f"{row.uid}.npy")
        pose, pose_length = preprocess_array(
            raw, max_length=self.cfg["max_pose_length"],
            normalization=self.cfg["normalization"], pose_dim=self.pose_dim,
            use_motion=self.use_motion)
        enc = self.tok(row.text, max_length=self.cfg["max_text_length"], padding="max_length",
                       truncation=True, return_tensors="pt")
        labels = enc["input_ids"].squeeze(0).clone()
        labels[labels == self.tok.pad_token_id] = -100
        return {"pose": torch.from_numpy(pose).float(), "pose_length": torch.tensor(pose_length),
                "labels": labels, "input_ids": enc["input_ids"].squeeze(0),
                "attention_mask": enc["attention_mask"].squeeze(0)}


def collate_fn(batch):
    return {k: torch.stack([b[k] for b in batch]) for k in batch[0]}


def make_loaders(train_df, features_dir, tokenizer, variant, cfg):
    """Their create_data_loaders: random_split 80/10/10 with Generator(seed)."""
    v = VARIANTS[variant]
    ds = PoseSLTDataset(train_df, features_dir, tokenizer, v["pose_dim"], v["use_motion"], cfg)
    total = len(ds)
    test_size = max(1, int(total * 0.1))
    val_size = max(1, int(total * 0.1))
    train_size = total - test_size - val_size
    gen = torch.Generator().manual_seed(cfg["seed"])
    train_set, val_set, _unused = random_split(ds, [train_size, val_size, test_size],
                                               generator=gen)
    kw = dict(batch_size=cfg["batch_size"], num_workers=cfg["num_workers"],
              collate_fn=collate_fn, pin_memory=torch.cuda.is_available(),
              persistent_workers=cfg["num_workers"] > 0)
    return {"train": DataLoader(train_set, shuffle=True, **kw),
            "val": DataLoader(val_set, shuffle=False, **kw)}


# ------------------------------------------------------------------------- training

def train(variant, train_df, features_dir, out_dir, cfg=None, init_from=None,
          max_steps=None):
    """Train one variant. Returns the path of best_model.
    init_from: optional checkpoint dir (e.g. the paper's released weights) to
    start from instead of the plain pretrained T5."""
    cfg = {**TRAIN_DEFAULTS, **(cfg or {})}
    v = VARIANTS[variant]
    torch.manual_seed(cfg["seed"])
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    tokenizer = AutoTokenizer.from_pretrained(v["base_model"])
    loaders = make_loaders(train_df, features_dir, tokenizer, variant, cfg)
    model = PoseT5(v["base_model"], pose_dim=v["pose_dim"]).to(device)
    if init_from:
        model.load_pretrained(str(init_from))
        model.to(device)
    optimizer = AdamW(model.parameters(), lr=float(cfg["learning_rate"]))

    amp = cfg["mixed_precision"]
    amp_on = device.type == "cuda" and amp in {"fp16", "bf16"}
    amp_dtype = torch.float16 if amp == "fp16" else torch.bfloat16
    scaler = torch.amp.GradScaler("cuda", enabled=amp_on and amp == "fp16")
    accum = max(1, cfg["gradient_accumulation_steps"])

    logger.info("variant=%s device=%s train=%d val=%d epochs=%d", variant, device,
                len(loaders["train"].dataset), len(loaders["val"].dataset), v["epochs"])

    def evaluate_loss():
        model.eval()
        total = 0.0
        with torch.no_grad():
            for b in loaders["val"]:
                out = model(pose=b["pose"].to(device), pose_length=b["pose_length"].to(device),
                            input_ids=b["input_ids"].to(device),
                            attention_mask=b["attention_mask"].to(device),
                            labels=b["labels"].to(device))
                total += out.loss.item()
        model.train()
        return total / len(loaders["val"])

    global_step, best, bad_loss_streak = 0, float("inf"), 0
    history = []
    model.train()
    t0 = time.time()
    for epoch in range(v["epochs"]):
        running = 0.0
        n_batches = len(loaders["train"])
        optimizer.zero_grad(set_to_none=True)
        for batch_idx, b in enumerate(loaders["train"]):
            with torch.autocast(device_type=device.type, dtype=amp_dtype, enabled=amp_on):
                out = model(pose=b["pose"].to(device), pose_length=b["pose_length"].to(device),
                            input_ids=b["input_ids"].to(device),
                            attention_mask=b["attention_mask"].to(device),
                            labels=b["labels"].to(device))
                loss = out.loss / accum
            scaler.scale(loss).backward()
            if (batch_idx + 1) % accum == 0 or batch_idx + 1 == n_batches:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), cfg["max_grad_norm"])
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)

            step_loss = loss.item() * accum
            bad_loss_streak = bad_loss_streak + 1 if not math.isfinite(step_loss) else 0
            if bad_loss_streak == 50:
                logger.warning("loss has been NaN/inf for 50 steps: fp16 overflow. "
                               "Re-run with cfg={'mixed_precision': 'no'}.")
            running += step_loss if math.isfinite(step_loss) else 0.0
            global_step += 1

            if global_step % cfg["logging_steps"] == 0:
                logger.info("epoch %d step %d  train_loss %.4f  (%.1f min)", epoch + 1,
                            global_step, running / (batch_idx + 1), (time.time() - t0) / 60)
            if global_step % cfg["eval_steps"] == 0:
                val = evaluate_loss()
                history.append({"step": global_step, "epoch": epoch + 1, "val_loss": val})
                if val < best:
                    best = val
                    model.save_pretrained(str(out_dir / "best_model"))
                    logger.info("step %d  val_loss %.4f  -> new best_model", global_step, val)
                else:
                    logger.info("step %d  val_loss %.4f", global_step, val)
            if max_steps and global_step >= max_steps:
                break
        logger.info("epoch %d done, train_loss %.4f", epoch + 1, running / max(1, batch_idx + 1))
        if max_steps and global_step >= max_steps:
            break

    # Also evaluate at the very end so short runs still produce a best_model.
    val = evaluate_loss()
    history.append({"step": global_step, "epoch": "end", "val_loss": val})
    if val < best or not (out_dir / "best_model").exists():
        model.save_pretrained(str(out_dir / "best_model"))
    model.save_pretrained(str(out_dir / "final_model"))
    json.dump(history, open(out_dir / "val_history.json", "w"), indent=1)
    return out_dir / "best_model"


# ----------------------------------------------------------------------- evaluation

def evaluate(variant, checkpoint, test_df, features_dir, out_dir, cfg=None, limit=None):
    """Their evaluate_all.evaluate_model: BLEU, chrF, ROUGE-1/2/L + predictions.csv."""
    from rouge_score import rouge_scorer
    from sacrebleu.metrics import BLEU, CHRF

    cfg = {**TRAIN_DEFAULTS, **(cfg or {})}
    v = VARIANTS[variant]
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    tokenizer = AutoTokenizer.from_pretrained(v["base_model"])
    df = test_df if limit is None else test_df.head(limit)
    ds = PoseSLTDataset(df, features_dir, tokenizer, v["pose_dim"], v["use_motion"], cfg)
    loader = DataLoader(ds, batch_size=cfg["batch_size"], shuffle=False,
                        num_workers=cfg["num_workers"], collate_fn=collate_fn)

    model = PoseT5(v["base_model"], pose_dim=v["pose_dim"],
                   local_model_path=str(Path(checkpoint) / "t5")).to(device)
    model.load_pretrained(str(checkpoint))
    model.to(device).eval()

    preds = []
    with torch.no_grad():
        for b in loader:
            ids = model.generate(pose=b["pose"].to(device), pose_length=b["pose_length"].to(device),
                                 max_length=cfg["max_gen_length"], num_beams=cfg["beam_size"])
            preds.extend(tokenizer.batch_decode(ids, skip_special_tokens=True))
    preds = [p.strip() for p in preds]
    refs = [r.strip() for r in df["text"].tolist()]

    bleu = BLEU().corpus_score(preds, [refs]).score
    chrf = CHRF().corpus_score(preds, [refs]).score
    scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)
    scores = [scorer.score(r, p) for r, p in zip(refs, preds)]
    rouge = {k: sum(s[k].fmeasure for s in scores) / len(scores)
             for k in ("rouge1", "rouge2", "rougeL")}

    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    with open(out_dir / "predictions.csv", "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=["uid", "reference", "prediction"])
        w.writeheader()
        w.writerows({"uid": u, "reference": r, "prediction": p}
                    for u, r, p in zip(df["uid"], refs, preds))
    metrics = {"variant": variant, "checkpoint": str(checkpoint), "test_examples": len(refs),
               "BLEU": bleu, "chrF": chrf, "ROUGE-1": rouge["rouge1"] * 100,
               "ROUGE-2": rouge["rouge2"] * 100, "ROUGE-L": rouge["rougeL"] * 100,
               "unique_predictions": len(set(preds))}
    json.dump(metrics, open(out_dir / "metrics.json", "w"), indent=2)
    return metrics

### Configuration

In [ ]:
import glob, json, logging, os, sys
import pandas as pd, torch
sys.path.insert(0, ".")
import pipeline
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(message)s", force=True)

VARIANT = "t5-small-motion"   # "t5-small" | "t5-small-motion" | "t5-base" | "t5-large"
SPLIT = "paper"               # "paper" = exact paper split | "video" = stricter, no shared videos
MIXED_PRECISION = "fp16"      # paper setting; switch to "no" if the log warns about NaN loss
RUN_RELEASED_BASELINE = True  # also score the authors' released checkpoint on the same split

csv_path = glob.glob("/kaggle/input/**/iSign_v1.1.csv", recursive=True)[0]
FEATURES = os.path.dirname(csv_path)
WORK = "/kaggle/working"
print("features:", FEATURES, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")

### Step 1 · data and splits

In [ ]:
df = pipeline.load_annotations(csv_path, FEATURES)
split_fn = pipeline.paper_test_split if SPLIT == "paper" else pipeline.video_test_split
train_df, test_df = split_fn(df)
print(f"usable samples {len(df):,} -> train {len(train_df):,} / test {len(test_df):,}  "
      "(paper: 18,867 -> 16,980 / 1,887)")
test_df.to_csv(f"{WORK}/test_split_{SPLIT}.csv", index=False)
test_df.head()

### Step 2 (optional) · score the authors' released checkpoint on this test split
If our data and split match the paper's, this should land near their **0.298 BLEU** for
t5-small-motion. It is a check that the reproduction is set up correctly.

In [ ]:
if RUN_RELEASED_BASELINE:
    from huggingface_hub import snapshot_download
    rel = snapshot_download("manavdhamecha77/iSign-t5-pose-to-text",
                            allow_patterns=[f"{VARIANT}/pose_encoder.pt", f"{VARIANT}/t5/*"])
    released = pipeline.evaluate(VARIANT, os.path.join(rel, VARIANT), test_df, FEATURES,
                                 f"{WORK}/eval_released_{VARIANT}")
    print(json.dumps(released, indent=2))

### Step 3 · train (exact paper recipe)

In [ ]:
best = pipeline.train(VARIANT, train_df, FEATURES, f"{WORK}/checkpoints/{VARIANT}",
                      cfg={"mixed_precision": MIXED_PRECISION})
print("best model:", best)

### Step 4 · evaluate on the held-out test split

In [ ]:
metrics = pipeline.evaluate(VARIANT, best, test_df, FEATURES, f"{WORK}/eval_{VARIANT}")
print(json.dumps(metrics, indent=2))
pred = pd.read_csv(f"{WORK}/eval_{VARIANT}/predictions.csv")
pd.set_option("display.max_colwidth", 80)
pred.sample(20, random_state=0)

### Step 5 · results table (paper vs. ours)

In [ ]:
paper = {"t5-small": 0.189, "t5-base": 0.154, "t5-large": 0.132, "t5-small-motion": 0.298}
rows = [{"model": f"{VARIANT} (paper, Table 1)", "BLEU": paper[VARIANT]},
        {"model": f"{VARIANT} (ours)", **{k: round(metrics[k], 3) for k in
                                          ("BLEU", "chrF", "ROUGE-1", "ROUGE-2", "ROUGE-L")}}]
if RUN_RELEASED_BASELINE:
    rows.insert(1, {"model": f"{VARIANT} (released ckpt, our split)",
                    **{k: round(released[k], 3) for k in ("BLEU", "chrF", "ROUGE-1", "ROUGE-2", "ROUGE-L")}})
table = pd.DataFrame(rows)
table.to_csv(f"{WORK}/results_{VARIANT}.csv", index=False)
table

### Getting the model onto your laptop
Everything is in the version's **Output** tab: `checkpoints/<variant>/best_model/`
(`pose_encoder.pt` + `t5/`), predictions and metrics. Download `best_model`, or save the output as
a dataset. It loads with `PoseT5(...).load_pretrained(best_model)`, the same format as the
paper's HuggingFace checkpoints.